In [1]:
# Imports + Paths

import os
import time
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications import vgg19
from tensorflow.keras.preprocessing.image import load_img, img_to_array
import matplotlib.pyplot as plt

tf.random.set_seed(42)
np.random.seed(42)

ROOT_DIR = r"E:\JYU\Deep-Learning for Cognitive Computing for Developers\Task 5\Task5-2_StyleTransfer"

CONTENT_DIR = os.path.join(ROOT_DIR, "content_images")
STYLE_DIR   = os.path.join(ROOT_DIR, "style_images")
OUT_DIR     = os.path.join(ROOT_DIR, "outputs")

os.makedirs(OUT_DIR, exist_ok=True)

content_files = [f for f in os.listdir(CONTENT_DIR) if f.lower().endswith((".jpg",".jpeg",".png"))]
style_files   = [f for f in os.listdir(STYLE_DIR) if f.lower().endswith((".jpg",".jpeg",".png"))]

print("TensorFlow:", tf.__version__)
print("Content images found:", len(content_files))
print("Style images found:", len(style_files))
print("Outputs directory ready.")

TensorFlow: 2.20.0
Content images found: 5
Style images found: 5
Outputs directory ready.


In [2]:
# 1 content + 1 style image and preview

import os
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing.image import load_img, img_to_array

# List images
content_files = sorted([f for f in os.listdir(CONTENT_DIR) if f.lower().endswith((".jpg",".jpeg",".png"))])
style_files   = sorted([f for f in os.listdir(STYLE_DIR) if f.lower().endswith((".jpg",".jpeg",".png"))])

# Choose index (change these anytime)
CONTENT_IDX = 0
STYLE_IDX   = 0

content_path = os.path.join(CONTENT_DIR, content_files[CONTENT_IDX])
style_path   = os.path.join(STYLE_DIR, style_files[STYLE_IDX])

print("Selected content:", content_files[CONTENT_IDX])
print("Selected style  :", style_files[STYLE_IDX])

# Load for preview (keep moderate size)
MAX_DIM = 512

def load_and_resize(path, max_dim=512):
    img = load_img(path)
    w, h = img.size
    scale = max_dim / max(w, h)
    new_w, new_h = int(w * scale), int(h * scale)
    img = img.resize((new_w, new_h))
    arr = img_to_array(img)
    return arr

content_img = load_and_resize(content_path, MAX_DIM).astype("uint8")
style_img   = load_and_resize(style_path, MAX_DIM).astype("uint8")

plt.figure(figsize=(10,4))
plt.subplot(1,2,1); plt.imshow(content_img); plt.title("Content"); plt.axis("off")
plt.subplot(1,2,2); plt.imshow(style_img);   plt.title("Style");   plt.axis("off")
plt.show()

Selected content: image1.jpg
Selected style  : image10.jpg


<Figure size 1000x400 with 2 Axes>

In [3]:
# Load VGG19 (pretrained ImageNet, no top)

vgg = vgg19.VGG19(weights="imagenet", include_top=False)
vgg.trainable = False

print("VGG19 loaded.")
print("Total layers:", len(vgg.layers))

# Show some key layer names (important for style/content extraction)
for layer in vgg.layers[:10]:
    print(layer.name)

VGG19 loaded.
Total layers: 22
input_layer
block1_conv1
block1_conv2
block1_pool
block2_conv1
block2_conv2
block2_pool
block3_conv1
block3_conv2
block3_conv3


In [4]:
# Define content and style layers

# Content layer (high-level structure)
CONTENT_LAYERS = ['block5_conv2']

# Style layers (multiple levels for texture & color)
STYLE_LAYERS = [
    'block1_conv1',
    'block2_conv1',
    'block3_conv1',
    'block4_conv1',
    'block5_conv1'
]

print("Content layers:", CONTENT_LAYERS)
print("Style layers:", STYLE_LAYERS)

Content layers: ['block5_conv2']
Style layers: ['block1_conv1', 'block2_conv1', 'block3_conv1', 'block4_conv1', 'block5_conv1']


In [5]:
# Build VGG feature extraction model (content + style outputs)

# Collect outputs from selected layers
content_outputs = [vgg.get_layer(name).output for name in CONTENT_LAYERS]
style_outputs   = [vgg.get_layer(name).output for name in STYLE_LAYERS]

# Combine
model_outputs = style_outputs + content_outputs

feature_extractor = tf.keras.Model(inputs=vgg.input, outputs=model_outputs)

feature_extractor.trainable = False

print("Feature extractor built.")
print("Total outputs:", len(model_outputs))

Feature extractor built.
Total outputs: 6


In [6]:
# Helper functions: preprocess, deprocess, gram matrix

def preprocess_image(path, max_dim=512):
    img = load_img(path)
    w, h = img.size
    scale = max_dim / max(w, h)
    new_w, new_h = int(w * scale), int(h * scale)
    img = img.resize((new_w, new_h))
    img = img_to_array(img)
    img = np.expand_dims(img, axis=0)
    img = vgg19.preprocess_input(img)
    return tf.convert_to_tensor(img, dtype=tf.float32)

def deprocess_image(processed):
    x = processed.numpy()
    if len(x.shape) == 4:
        x = x[0]
    # Undo VGG19 preprocess: BGR + mean subtraction
    x[:, :, 0] += 103.939
    x[:, :, 1] += 116.779
    x[:, :, 2] += 123.68
    x = x[:, :, ::-1]  # BGR -> RGB
    x = np.clip(x, 0, 255).astype("uint8")
    return x

def gram_matrix(tensor):
    # tensor: (1, h, w, c)
    x = tensor[0]
    x = tf.reshape(x, (-1, x.shape[-1]))  # (h*w, c)
    n = tf.shape(x)[0]
    gram = tf.matmul(x, x, transpose_a=True) / tf.cast(n, tf.float32)
    return gram

In [7]:
# Extract style + content targets from selected images

content_tensor = preprocess_image(content_path, max_dim=512)
style_tensor   = preprocess_image(style_path, max_dim=512)

outputs_style_content = feature_extractor(content_tensor)
style_outputs_content = outputs_style_content[:len(STYLE_LAYERS)]
content_outputs_content = outputs_style_content[len(STYLE_LAYERS):]

outputs_style_style = feature_extractor(style_tensor)
style_outputs_style = outputs_style_style[:len(STYLE_LAYERS)]

# Targets
content_target = content_outputs_content[0]

style_targets = [gram_matrix(o) for o in style_outputs_style]

print("Content target shape:", content_target.shape)
print("Number of style targets:", len(style_targets))
print("First style gram shape:", style_targets[0].shape)

Content target shape: (1, 32, 18, 512)
Number of style targets: 5
First style gram shape: (64, 64)


In [8]:
# Initialize generated image (start from content image)

generated = tf.Variable(content_tensor)

print("Generated initialized:", generated.shape, generated.dtype)

Generated initialized: (1, 512, 288, 3) <dtype: 'float32'>


In [9]:
# Loss functions

def compute_content_loss(gen_content, content_target):
    return tf.reduce_mean(tf.square(gen_content - content_target))

def compute_style_loss(gen_style_outputs, style_targets):
    style_loss = 0.0
    for gen_out, target in zip(gen_style_outputs, style_targets):
        gram_gen = gram_matrix(gen_out)
        style_loss += tf.reduce_mean(tf.square(gram_gen - target))
    return style_loss / float(len(style_targets))

def total_loss(gen_outputs, style_targets, content_target, style_weight=1e-2, content_weight=1e4):
    gen_style = gen_outputs[:len(STYLE_LAYERS)]
    gen_content = gen_outputs[len(STYLE_LAYERS):][0]

    s_loss = compute_style_loss(gen_style, style_targets)
    c_loss = compute_content_loss(gen_content, content_target)

    loss = style_weight * s_loss + content_weight * c_loss
    return loss, s_loss, c_loss

print("Loss functions ready.")

Loss functions ready.


In [10]:
# Optimize generated image

optimizer = tf.keras.optimizers.Adam(learning_rate=0.02)

EPOCHS = 200     # you can change to 300 later for better quality
print_every = 50

best_loss = float("inf")
best_img = None

start = time.time()

for i in range(1, EPOCHS + 1):
    with tf.GradientTape() as tape:
        outputs = feature_extractor(generated)
        loss, s_loss, c_loss = total_loss(
            outputs,
            style_targets,
            content_target,
            style_weight=1e-1,
            content_weight=1e3
        )

    grads = tape.gradient(loss, generated)
    optimizer.apply_gradients([(grads, generated)])

    # Keep pixel values in a reasonable range for VGG preprocessing space
    generated.assign(tf.clip_by_value(generated, -128.0, 128.0))

    if loss < best_loss:
        best_loss = float(loss.numpy())
        best_img = deprocess_image(generated)

    if i % print_every == 0 or i == 1:
        print(f"iter {i:04d} | total={loss.numpy():.4f} | style={s_loss.numpy():.6f} | content={c_loss.numpy():.6f}")

end = time.time()
print("\nDone. Seconds:", round(end - start, 2))

iter 0001 | total=909263296.0000 | style=9092632576.000000 | content=0.000000
iter 0050 | total=620219072.0000 | style=6200886272.000000 | content=130.428116
iter 0100 | total=447077408.0000 | style=4467149824.000000 | content=362.434570
iter 0150 | total=340336160.0000 | style=3397046016.000000 | content=631.552490
iter 0200 | total=274453024.0000 | style=2735675648.000000 | content=885.447021

Done. Seconds: 1117.05


In [11]:
# Save and visualize result

out_name = f"styled_c{CONTENT_IDX}_s{STYLE_IDX}_e{EPOCHS}.png"
out_path = os.path.join(OUT_DIR, out_name)

plt.imsave(out_path, best_img)

print("Saved:", out_path)

# Show comparison grid
content_preview = load_and_resize(content_path, 512).astype("uint8")
style_preview   = load_and_resize(style_path, 512).astype("uint8")

plt.figure(figsize=(15,5))
plt.subplot(1,3,1); plt.imshow(content_preview); plt.title("Content"); plt.axis("off")
plt.subplot(1,3,2); plt.imshow(style_preview);   plt.title("Style");   plt.axis("off")
plt.subplot(1,3,3); plt.imshow(best_img);        plt.title("Output");  plt.axis("off")
plt.show()

Saved: E:\JYU\Deep-Learning for Cognitive Computing for Developers\Task 5\Task5-2_StyleTransfer\outputs\styled_c0_s0_e200.png


<Figure size 1500x500 with 3 Axes>

In [12]:
# Batch generate 3 content × 3 style (150 epochs each)

STYLE_W = 5.0
CONTENT_W = 1e3
EPOCHS_BATCH = 150
PRINT_EVERY = 50
MAX_DIM = 512

content_files = sorted([f for f in os.listdir(CONTENT_DIR) if f.lower().endswith((".jpg",".jpeg",".png"))])
style_files   = sorted([f for f in os.listdir(STYLE_DIR) if f.lower().endswith((".jpg",".jpeg",".png"))])

content_indices = [0, 1, 2]
style_indices   = [0, 1, 2]

saved_paths = []

def run_style_transfer(content_path, style_path, epochs=100):
    optimizer = tf.keras.optimizers.Adam(learning_rate=0.02)

    content_tensor = preprocess_image(content_path, max_dim=MAX_DIM)
    style_tensor   = preprocess_image(style_path, max_dim=MAX_DIM)

    out_c = feature_extractor(content_tensor)
    content_target = out_c[len(STYLE_LAYERS):][0]

    out_s = feature_extractor(style_tensor)
    style_targets = [gram_matrix(o) for o in out_s[:len(STYLE_LAYERS)]]

    generated = tf.Variable(content_tensor)

    best_loss = float("inf")
    best_img = None

    for i in range(1, epochs + 1):
        with tf.GradientTape() as tape:
            outputs = feature_extractor(generated)
            loss, s_loss, c_loss = total_loss(
                outputs, style_targets, content_target,
                style_weight=STYLE_W,
                content_weight=CONTENT_W
            )

        grads = tape.gradient(loss, generated)
        optimizer.apply_gradients([(grads, generated)])
        generated.assign(tf.clip_by_value(generated, -128.0, 128.0))

        if loss < best_loss:
            best_loss = float(loss.numpy())
            best_img = deprocess_image(generated)

        if i % PRINT_EVERY == 0 or i == 1:
            print(f"  iter {i:04d} | total={loss.numpy():.2f} | style={s_loss.numpy():.2f} | content={c_loss.numpy():.2f}")

    return best_img

start_all = time.time()

for ci in content_indices:
    for si in style_indices:
        out_name = f"batch_c{ci}_s{si}_e{EPOCHS_BATCH}.png"
        out_path = os.path.join(OUT_DIR, out_name)

        # Skip if already generated
        if os.path.exists(out_path):
            print("SKIP (exists):", out_name)
            saved_paths.append(out_path)
            continue

        c_path = os.path.join(CONTENT_DIR, content_files[ci])
        s_path = os.path.join(STYLE_DIR, style_files[si])

        print("\n=== Running:", f"content[{ci}]={content_files[ci]}", "|", f"style[{si}]={style_files[si]}", "===")
        t0 = time.time()
        best_img = run_style_transfer(c_path, s_path, epochs=EPOCHS_BATCH)
        t1 = time.time()

        plt.imsave(out_path, best_img)
        print("Saved:", out_path, "| seconds:", round(t1 - t0, 2))
        saved_paths.append(out_path)

end_all = time.time()
print("\nALL DONE. Total seconds:", round(end_all - start_all, 2))
print("Total images available:", len(saved_paths))


=== Running: content[0]=image1.jpg | style[0]=image10.jpg ===
  iter 0001 | total=45463162880.00 | style=9092632576.00 | content=0.00
  iter 0050 | total=31003535360.00 | style=6200680960.00 | content=131.06
  iter 0100 | total=22331684864.00 | style=4466264064.00 | content=365.43
  iter 0150 | total=16976985088.00 | style=3395269120.00 | content=639.59
Saved: E:\JYU\Deep-Learning for Cognitive Computing for Developers\Task 5\Task5-2_StyleTransfer\outputs\batch_c0_s0_e150.png | seconds: 841.97

=== Running: content[0]=image1.jpg | style[1]=image6.jpg ===
  iter 0001 | total=32160137216.00 | style=6432027648.00 | content=0.00
  iter 0050 | total=21463107584.00 | style=4292577792.00 | content=218.61
  iter 0100 | total=14243519488.00 | style=2848602880.00 | content=505.31
  iter 0150 | total=10120519680.00 | style=2023960832.00 | content=715.59
Saved: E:\JYU\Deep-Learning for Cognitive Computing for Developers\Task 5\Task5-2_StyleTransfer\outputs\batch_c0_s1_e150.png | seconds: 842.19



In [13]:
# Create a 3×3 grid image of outputs

import matplotlib.pyplot as plt

content_indices = [0, 1, 2]
style_indices   = [0, 1, 2]
EPOCHS_BATCH = 100

fig, axes = plt.subplots(3, 3, figsize=(12, 12))

for r, ci in enumerate(content_indices):
    for c, si in enumerate(style_indices):
        fname = f"batch_c{ci}_s{si}_e{EPOCHS_BATCH}.png"
        fpath = os.path.join(OUT_DIR, fname)
        img = plt.imread(fpath)
        axes[r, c].imshow(img)
        axes[r, c].axis("off")
        axes[r, c].set_title(f"c{ci}-s{si}", fontsize=10)

plt.tight_layout()

grid_path = os.path.join(OUT_DIR, "style_transfer_grid_3x3.png")
plt.savefig(grid_path, dpi=200)
print("Saved grid:", grid_path)

plt.show()

Saved grid: E:\JYU\Deep-Learning for Cognitive Computing for Developers\Task 5\Task5-2_StyleTransfer\outputs\style_transfer_grid_3x3.png


<Figure size 1200x1200 with 9 Axes>

## Neural Style Transfer — Theoretical Explanation

Neural Style Transfer (NST) uses a pretrained convolutional neural network (VGG19) to separate and recombine image content and artistic style.


__Content Representation__

Content features are extracted from deeper convolutional layers (for example, block5_conv2).
These layers preserve high-level spatial structure such as object layout and shapes.

Content loss is computed as the squared difference between feature maps of:
- the generated image
- the original content image

This ensures the generated image keeps the original structure.


__Style Representation__

Style features are extracted from multiple convolutional layers.

To represent style, we compute the Gram matrix, which measures correlations between feature channels.
The Gram matrix captures:
- texture
- brush strokes
- color distribution
- artistic patterns

Style loss is the squared difference between:
- Gram matrix of generated image
- Gram matrix of style image


__Total Loss Function__

The total loss is a weighted combination of:
Total Loss = (alpha × Style Loss) + (beta × Content Loss)

Where:
- alpha controls how strong the artistic style is
- beta controls how much original structure is preserved

By optimizing the pixel values of the generated image using gradient descent, we produce an image that maintains content while adopting the artistic style.

## Experimental Analysis

In this experiment, VGG19 pretrained on ImageNet was used as a fixed feature extractor.

The content layer (block5_conv2) preserved high-level spatial structure, while five convolutional layers were used for style extraction.

__Observations__
1. Increasing the style weight (α) strengthened artistic texture and color transfer.
2. Increasing the content weight (β) preserved structural details but reduced stylistic intensity.
3. With low style weight, outputs were visually similar to the original content image.
4. With higher style weight (α = 5.0) and 150 optimization iterations, clear stylistic differences appeared across different style images.
5. Different style images produced noticeably distinct textures and color tones, validating the effectiveness of Gram matrix–based style representation.

__Computational Consideration__
- Each 150-epoch run required significant CPU time (~14–18 minutes per pair).
- Batch generation of 9 combinations required extended runtime.
- This demonstrates the computational cost of pixel-level optimization methods compared to feed-forward style transfer models.

Overall, the experiment confirms that neural style transfer successfully separates content and style representations and recombines them through optimization.